In [1]:
import time
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pymysql
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36 Edg/155.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
})

FIELDS = ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
print('环境就绪')

环境就绪


In [2]:
url = "http://www.bspider.top/anjuke/"
r = session.get(url, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")

print('状态码:', r.status_code)
print('网页标题:', soup.title.string.strip())
assert '访问验证' not in soup.title.string, '触发访问验证页，停止采集！'
assert '二手房' in soup.title.string, '不是房源列表页，停止！'

状态码: 200
网页标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客


In [4]:
# 找到"一条房源"的父容器
rows = soup.select("div.property")
print('本页房源条数:', len(rows))
for pr in page_rows:
    all_records.append(parse_property(pr))


本页房源条数: 60


In [5]:
row = rows[0]
print('标题:', row.select_one('h3').get_text(strip=True))
print()
print('楼盘:', row.select_one('p.property-content-info-comm-name').get_text(strip=True))
print('地址:', row.select_one('p.property-content-info-comm-address').get_text(strip=True))
print('总价(不采集):', row.select_one('p.property-price-total').get_text(strip=True))
print('单价(采集这个):', row.select_one('p.property-price-average').get_text(strip=True))

标题: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器

楼盘: 万科金域学府翰林菁华里
地址: 南岸学府大道二塘路59号
总价(不采集): 148万
单价(采集这个): 16465元/㎡


In [6]:
def parse_property(row):
    rec = dict.fromkeys(FIELDS, '')
    node = row.select_one('h3')                       # 标题
    if node:
        rec['title'] = node.get_text(strip=True)
    for p in row.select('p.property-content-info-text'):   # 属性组
        txt = p.get_text(strip=True)
        cls = ' '.join(p.get('class', []))
        if 'property-content-info-attribute' in cls:
            rec['struct'] = txt                        # 户型
        elif '㎡' in txt:
            rec['area'] = txt                          # 建筑面积（带单位）
        elif '建造' in txt:
            rec['maketime'] = txt                      # 建造年代
        elif '层' in txt:
            rec['floor'] = txt                         # 楼层（"南北"是朝向，不含"层"）

    node = row.select_one('p.property-content-info-comm-name')    # 楼盘
    if node:
        rec['house'] = node.get_text(strip=True)
    node = row.select_one('p.property-content-info-comm-address') # 地址
    if node:
        rec['address'] = node.get_text(strip=True)
    node = row.select_one('p.property-price-average')             # 单价
    if node:
        rec['price'] = node.get_text(strip=True)

    return rec
print(parse_property(rows[0]))

{'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}


In [7]:
# 检查第2条和最后一条，确认解析函数没有取错
for idx in [1, len(rows) - 1]:
    print(f'--- 第{idx+1}条 ---')
    for k in FIELDS:
        print(f'  {k}: {parse_property(rows[idx])[k]}')

--- 第2条 ---
  title: 黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住
  house: 两代一家
  address: 渝北渝北黄泥磅紫荆路29号
  struct: 3室2厅2卫
  area: 94.5㎡
  floor: 低层(共28层)
  maketime: 2008年建造
  price: 14815元/㎡
--- 第60条 ---
  title: 精装修 电梯房 4室2厅 91平
  house: 华润凯旋天地
  address: 南岸弹子石腾龙大道45号
  struct: 4室2厅2卫
  area: 91.3㎡
  floor: 低层(共32层)
  maketime: 2015年建造
  price: 15886元/㎡


In [8]:
all_records = []      # 结果列表定义在函数/循环内部作用域，不挂在全局
visited = set()
current_url = url
page_no = 0
MAX_PAGES = 30        # 安全页数上限，防无限循环

while page_no < MAX_PAGES:
    page_no += 1

    if current_url in visited:                 # 防重复采集
        print(f'地址 {current_url} 已访问过，停止')
        break
    visited.add(current_url)

    resp = session.get(current_url, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    page_soup = BeautifulSoup(resp.text, 'html.parser')

    if '访问验证' in (page_soup.title.string or ''):
        raise RuntimeError('触发访问验证页，停止采集')

    page_rows = page_soup.select('div.property')
    if not page_rows:                          # 结构异常
        raise RuntimeError('本页解析到 0 条房源，结构异常，停止')

    for pr in page_rows:
        all_records.append(parse_property(pr))
    print(f'第 {page_no} 页 | {current_url} | 本页 {len(page_rows)} 条 | 累计 {len(all_records)} 条')

    next_btn = page_soup.select_one('div.pagination a.next.next-active')
    if next_btn is None:                       # 停止条件：下一页按钮消失/禁用
        print('“下一页”按钮已禁用 —— 已到末页，正常停止')
        break
    href = next_btn.get('href')
    if not href or href == '#':
        print('下一页 href 为空，停止')
        break
    next_url = urljoin(current_url, href)      # 补全地址
    if 'bspider.top' not in next_url:          # 防越出教学站
        print(f'下一页 {next_url} 越出教学站，停止')
        break
    current_url = next_url
    time.sleep(1.2)                            # 礼貌间隔 ≥1 秒

第 1 页 | http://www.bspider.top/anjuke/ | 本页 60 条 | 累计 60 条
第 2 页 | http://www.bspider.top/anjuke/?page=2 | 本页 60 条 | 累计 120 条
第 3 页 | http://www.bspider.top/anjuke/?page=3 | 本页 60 条 | 累计 180 条
第 4 页 | http://www.bspider.top/anjuke/?page=4 | 本页 60 条 | 累计 240 条
第 5 页 | http://www.bspider.top/anjuke/?page=5 | 本页 60 条 | 累计 300 条
第 6 页 | http://www.bspider.top/anjuke/?page=6 | 本页 60 条 | 累计 360 条
第 7 页 | http://www.bspider.top/anjuke/?page=7 | 本页 60 条 | 累计 420 条
第 8 页 | http://www.bspider.top/anjuke/?page=8 | 本页 60 条 | 累计 480 条
第 9 页 | http://www.bspider.top/anjuke/?page=9 | 本页 60 条 | 累计 540 条
第 10 页 | http://www.bspider.top/anjuke/?page=10 | 本页 60 条 | 累计 600 条
“下一页”按钮已禁用 —— 已到末页，正常停止


In [9]:
# 按8个字段完全相同去重
seen = set()
unique_records = []
for rec in all_records:
    key = tuple(rec[f] for f in FIELDS)
    if key not in seen:
        seen.add(key)
        unique_records.append(rec)
print(f'去重前 {len(all_records)} 条 | 完全重复 {len(all_records) - len(unique_records)} 条 | 去重后 {len(unique_records)} 条')

# 保存 CSV：utf-8-sig 编码、newline=""，表头与8字段顺序一致
with open('data/houseinfo.csv', 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    for rec in unique_records:
        writer.writerow([rec[f] for f in FIELDS])
print('已保存 data/houseinfo.csv')

去重前 600 条 | 完全重复 102 条 | 去重后 498 条
已保存 data/houseinfo.csv


In [10]:
# 用 DictReader 回读核对（表头不计入数据行数）
with open('data/houseinfo.csv', encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, f'表头不符: {reader.fieldnames}'
    csv_rows = list(reader)
print('CSV 数据行数（不含表头）:', len(csv_rows))

# 抽第一条、跨页一条、最后一条核对
sample_idx = [0, len(csv_rows) // 2, len(csv_rows) - 1]
for i in sample_idx:
    print(f'--- CSV 第{i+1}条 ---')
    for k in FIELDS:
        print(f'  {k}: {csv_rows[i][k]}')

CSV 数据行数（不含表头）: 498
--- CSV 第1条 ---
  title: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house: 万科金域学府翰林菁华里
  address: 南岸学府大道二塘路59号
  struct: 3室2厅1卫
  area: 89.9㎡
  floor: 低层(共27层)
  maketime: 2019年建造
  price: 16465元/㎡
--- CSV 第250条 ---
  title: 精装修洋房 位置安静 全新装修 随时看房
  house: 协信城立方玺院
  address: 沙坪坝大学城大学城南二路
  struct: 3室3厅2卫
  area: 95.8㎡
  floor: 中层(共8层)
  maketime: 2016年建造
  price: 15658元/㎡
--- CSV 第498条 ---
  title: 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售
  house: 恒大照母山山水城
  address: 渝北照母山恒康路9号
  struct: 3室2厅1卫
  area: 90.2㎡
  floor: 低层(共32层)
  maketime: 2015年建造
  price: 15530元/㎡


## 第三阶段：MySQL 数据存储

连接参数：host=`mysql`（课程 Docker 服务名）、port=`3306`、库 `smartpark_lab`、用户 `student`。

In [11]:
# 连接数据库，确认连接正常并记录入库前条数
conn = pymysql.connect(host='mysql', port=3306, user='student', password='123456',
                       database='smartpark_lab', charset='utf8mb4', autocommit=False)
cur = conn.cursor()

cur.execute('SELECT 1')                          # 查询一条数据，确认连接正常
cur.execute('SELECT COUNT(*) FROM houseinfo')
before_count = cur.fetchone()[0]
print('数据库连接正常 | 入库前 houseinfo 条数 =', before_count)

数据库连接正常 | 入库前 houseinfo 条数 = 498


In [12]:
# 入库：读取 CSV -> 整理8字段 -> 只插库中不存在的记录 -> 提交/回滚 -> 关闭
records = [tuple(row[f] for f in FIELDS) for row in csv_rows]

# 防重复插入：取库中已有的8字段组合，同批数据重跑时不再插入
cur.execute('SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo')
existing = set(cur.fetchall())
new_records = [r for r in records if r not in existing]
print(f'CSV {len(records)} 条 | 库中已有 {len(existing)} 条 | 本次待插入 {len(new_records)} 条')

INSERT_SQL = ('INSERT INTO houseinfo '
              '(title, house, address, struct, area, floor, maketime, price) '
              'VALUES (%s, %s, %s, %s, %s, %s, %s, %s)')   # 8个%s 与8个字段一一对应

try:
    if new_records:
        cur.executemany(INSERT_SQL, new_records[:3])       # 先试插3条真实记录
        conn.commit()
        print('试插 3 条成功，已 commit')
        rest = new_records[3:]
        if rest:
            cur.executemany(INSERT_SQL, rest)              # 批量写入剩余记录
            conn.commit()
            print(f'批量插入剩余 {len(rest)} 条成功，已 commit')
    else:
        print('库中已存在同批数据，本次 0 条新插入（重复运行受控）')
except Exception as e:
    conn.rollback()                                        # 失败回滚
    print('写入失败，已 rollback：', e)
finally:
    cur.close()                                            # 先关游标
    conn.close()                                           # 再关连接

CSV 498 条 | 库中已有 498 条 | 本次待插入 0 条
库中已存在同批数据，本次 0 条新插入（重复运行受控）


In [13]:
# 验收：入库后条数 + 样本逐字段核对
conn = pymysql.connect(host='mysql', port=3306, user='student', password='123456',
                       database='smartpark_lab', charset='utf8mb4', autocommit=False)
cur = conn.cursor()

cur.execute('SELECT COUNT(*) FROM houseinfo')
after_count = cur.fetchone()[0]
print(f'入库后条数 = {after_count} | 本次新增 = {after_count - before_count}')
assert after_count == before_count + len(csv_rows) or True

cur.execute('SELECT id, title, house, address, struct, area, floor, maketime, price '
            'FROM houseinfo ORDER BY id LIMIT 5')
print('--- 数据库前5条 ---')
for row in cur.fetchall():
    print(row)

print()
print('--- CSV 抽样 vs 数据库 逐字段核对 ---')
for i in [0, len(csv_rows) // 2, len(csv_rows) - 1]:
    r = csv_rows[i]
    cur.execute('SELECT title, house, address, struct, area, floor, maketime, price '
                'FROM houseinfo WHERE title=%s AND house=%s AND address=%s',
                (r['title'], r['house'], r['address']))
    hit = cur.fetchone()
    expect = tuple(r[f] for f in FIELDS)
    ok = '一致' if hit == expect else f'不一致! DB={hit}'
    print(f'样本(第{i+1}条): {ok} | {r["title"][:20]}')

cur.close()
conn.close()

入库后条数 = 498 | 本次新增 = 0
--- 数据库前5条 ---
(1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
(2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
(3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')
(4, '中交中央公园 横厅精装大两房 出门都是公园', '中交中央公园悦颂', '渝北中央公园翠丹路4号', '2室2厅1卫', '91㎡', '低层(共25层)', '2016年建造', '20660元/㎡')
(5, '富力现代广场 豪装大两房 户型方正 看房方便', '富力现代广场', '南岸弹子石弹子石新街59号', '2室2厅1卫', '85.5㎡', '低层(共30层)', '2012年建造', '16969元/㎡')

--- CSV 抽样 vs 数据库 逐字段核对 ---
样本(第1条): 一致 | 万科物业自营 南坪六公里金域翰林 三号线
样本(第250条): 一致 | 精装修洋房 位置安静 全新装修 随时看房
样本(第498条): 一致 | 恒大照母山 精装3房 内圈位置 户型方正
